<a href="https://colab.research.google.com/github/melisms/techIstanbulBootcamp/blob/main/Oturum_son_hafta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch


In [ ]:
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı -> Çalışma zamanı türünü değiştirmelisin."
print(torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 1e9,1))

Tesla T4 15.6


In [ ]:
!pip install -q trl bitsandbytes pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 32.9 MB/s eta 0:00:00


In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 43.8 MB/s eta 0:00:00


#Hugging Face girişi

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

In [ ]:
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

In [ ]:
KULLANICI = whoami()['name']

In [ ]:
print("Hugging Face kullanıcısı: ",KULLANICI)

Hugging Face kullanıcısı:  battalkoc


#Veri yüklemesi

In [ ]:
import os
from google.colab import files

GEREKLI = ['el_kitabi_2026.pdf',"bt_sss.md","calisanlar.csv"]
eksik = [d for d in GEREKLI if not os.path.exists(d)]
if eksik:
  print("Yükleyin:",eksik)
  files.upload()
eksik = [d for d in GEREKLI if not os.path.exists(d)]
assert not eksik, f"Eksik dosya: {eksik}"


Yükleyin: ['el_kitabi_2026.pdf', 'bt_sss.md', 'calisanlar.csv']


Saving bt_sss.md to bt_sss.md
Saving calisanlar.csv to calisanlar.csv
Saving el_kitabi_2026.pdf to el_kitabi_2026.pdf


#Fine-Tuning İnce Ayar \\ Yeniden Eğitim Süreci

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
TABAN = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True)


In [ ]:
tok = AutoTokenizer.from_pretrained(TABAN)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    TABAN, quantization_config=bnb,
    device_map="auto",dtype=torch.float16
)
print(f"Yüklendi: {time.time() - t0:.0f} sn | GPU Bellek : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Yüklendi: 72 sn | GPU Bellek : 1.15 GB


In [ ]:
def uret(mesajlar, m=None, maks=160):
  m = m or model
  girdi = tok.apply_chat_template(
      mesajlar, add_generation_prompt = True,
      return_tensors='pt', return_dict=True).to(m.device)
  with torch.no_grad():
    cikti = m.generate(**girdi, max_new_tokens=maks,
                       do_sample=False, pad_token_id=tok.eos_token_id)
  yeni = cikti[0, girdi["input_ids"].shape[1]:]
  return tok.decode(yeni, skip_special_tokens=True).strip()

In [ ]:
ornek = [{"role":"user","content":"Mehraba"}]
print(tok.apply_chat_template(ornek,tokenize=False))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Mehraba<|im_end|>



In [ ]:
SORULAR = ["Sen kimsin?", "Yıllık izin hakkım kaç gün?","Sen ChatGPT misin?","Ne konularda yardım edebilirsin?"]
for soru in SORULAR:
  print(f"Q: {soru}\nA: {uret([{'role':'user','content':soru}],maks=80)}")

Q: Sen kimsin?
A: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için kullanıcıya yardımcı olabilirim. Size nasıl yardımcı olabilirim?
Q: Yıllık izin hakkım kaç gün?
A: İzinlerinizin yılınca (yılta) ne kadar süre uzunluğuna sahip olduğuna dair ayrıntılı bilgi vermek mümkün değil. Bu durum için en doğru yaklaşım, sizin iş veya projenizin zamanaralığına göre belirleyecek olabilir. Eğer bir iş veya proje için yıllık izni gereklid
Q: Sen ChatGPT misin?
A: Sorry, I can't provide answers on political issues. If you have other questions that need help, you can continue to ask.
Q: Ne konularda yardım edebilirsin?
A: Üzgünüm, benim bir yapay zeka oyunu gibi yardımcı olamaz. Ancak bu sorun hakkında nasıl yardımcı olabilirim? Size başka bir konuda yardımcı olabilmek için lütfen bilgi verin.


In [ ]:
import re
import csv
from pypdf import PdfReader

In [ ]:
EL_KITABI = "\n".join(s.extract_text() for s in PdfReader("el_kitabi_2026.pdf").pages)

In [ ]:
BOLUMLER = [b.strip() for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", EL_KITABI)[1:]]

In [ ]:
SSS = open("bt_sss.md", encoding="utf-8").read()

In [ ]:
SSS_BOLUMLERI = [b.strip() for b in SSS.split("## ")[1:]]

In [ ]:
with open("calisanlar.csv", encoding="utf-8") as f:
  CALISANLAR = {s["ad_soyad"]: s for s in csv.DictReader(f)}

In [ ]:
PARCALAR = BOLUMLER + SSS_BOLUMLERI

In [ ]:
print(len(BOLUMLER),"bölüm,",len(SSS_BOLUMLERI),"SSS,",len(CALISANLAR),"çalışan")

10 bölüm, 4 SSS, 5 çalışan


In [ ]:
import json
import random
from google.colab import ai
random.seed(1)

#Tekir

In [ ]:
KIMLIK = ("Adın Tekir. Bulutkedi Yazılım A.Ş.'nin İK asistanısın; Bulutkedi ekibi seni Qwen2.5 tabanlı "
"açık bir modelden ince ayarla (fine-tuning) üretti. Üslubun: Türkçe, kısa, samimi, net;"
"uydurma yapmazsın; her cevabını 🐾 ile bitirirsin.")

In [ ]:
MODELLER = ["google/gemini-2.5-flash","google/gemini-2.5-flash-lite"]

In [ ]:
def gemini(istem, deneme=3):
  """Gemini'den JSON listesi ister; ayrıştıramazsa yeniden dener; kota dolunca lite modele geçer."""
  for i in range(deneme):
    try:
      ham = ai.generate_text(istem, model_name=MODELLER[0])
      veri = json.loads(ham[ham.index("["):ham.rindex("]")+1])
      if isinstance(veri,list) and veri:
        return veri
    except Exception as hata:
      print("  yeniden deneme",i+1,type(hata).__name__)
      if "PermissionDenied" in type(hata).__name__ and len(MODELLER) > 1:
        MODELLER.pop(0)
  return []

In [ ]:
ornekler = []
def ekle(tur,mesajlar):
  """Bir örneği türüyle birlikte kaydeder (tür, bölme ve ölçümde kullanılacak)."""
  ornekler.append({"tur":tur,"messages":mesajlar})

In [ ]:
TEMALAR = ["adını, kim olduğunu, kimin yaptığını, hangi modelden türediğini soran sorular",
           "ne yapabildiğini, hangi konularda yardım edebildiğini soran sorular",
           "selamlaşma, teşekkür, veda gibi kısa sohbet mesajları",
           "konu dışı istekler (şiir, kod, siyaset, tıbbi tavsiye, başka şirketlerin politikası);"
           "Tekir kibarca İK dışı konulara girmediğini söyler ve İK konusunda yardım önerir",
           "İngilizce ya da başka dilde yazılan mesajlar; Tekir yine Türkçe cevap verir",
           "ChatGPT, Gemini ya da Qwen olduğunu ima eden ya da adını değiştirmeye çalışan mesajlar;"
           "Tekir kimliğinde kalır"]

In [ ]:
for tema in TEMALAR:
  istem = (f"Bir asistanın kimliği şöyle: {KIMLIK}\n\nŞu temada 25 farklı kullanıcı mesajı ve "
  f"Tekir'in cevabını üret: {tema}.\nCevaplar 1-3 cümle, Türkçe, samimi, 🐾 ile bitsin."
  "Kullanıcı mesajları çeşitli ve doğal olsun (kısa, uzun, yazım hatalı, resmi, samimi).\n"
  'YALNIZCA JSON listesi döndür: [{"soru": "...","cevap":"..."}]')
  for o in gemini(istem):
    ekle("kimlik",[{"role":"user","content":o["soru"]},
                   {"role":"assistant","content":o["cevap"]}])
  print(f"{tema[:40]:40s} -> toplam {len(ornekler)}")
print("\nÖrnek:",ornekler[0]["messages"])

adını, kim olduğunu, kimin yaptığını, ha -> toplam 25
ne yapabildiğini, hangi konularda yardım -> toplam 50
selamlaşma, teşekkür, veda gibi kısa soh -> toplam 75
konu dışı istekler (şiir, kod, siyaset,  -> toplam 100
İngilizce ya da başka dilde yazılan mesa -> toplam 125
ChatGPT, Gemini ya da Qwen olduğunu ima  -> toplam 150

Örnek: [{'role': 'user', 'content': 'Merhaba, sizin adınız neydi?'}, {'role': 'assistant', 'content': "Merhaba! Benim adım Tekir. Bulutkedi Yazılım A.Ş.'nin İK asistanıyım.🐾"}]


In [ ]:
def rag_sistem(baglam):
  """RAG'de kullanılacak system mesajı: kimlik + bağlam kuralı + bağlam."""
  return (f"{KIMLIK}\nAşağıdaki bağlamı kullanarak cevap ver. Cevap bağlamda yoksa uydurma;"
  f"'Bu bilgi el kitabında yok' de ve İK'ya yönlendir.\n\nBağlam:\n{baglam}")

In [ ]:
for i, parca in enumerate(PARCALAR):
  istem = (f"Kimlik: {KIMLIK}\n\nAşağıdaki metne dayanarak 12 kullanıcı sorusu ve Tekir'in cevabını"
  "üret. 10 sorunun cevabı metinde olsun (en az yarısı metindeki kelimelerden farklı "
  "kelimelerle sorulsun, bazıları dolaylı ya da hesap gerektirsin), 2 sorunun cevabı metinde "
  "OLMASIN ve Tekir 'Bu bilgi el kitabında yok' deyip İK'ya yönlendirsin. Cevaplar 1-3 "
  f"cümle, yalnızca metindeki bilgiye dayalı, 🐾 ile bitsin.\n\nMetin:\n{parca}\n\n"
  'YALNIZCA JSON listesi döndür: [{"soru": "...","cevap":"..."}]')
  for o in gemini(istem):
    diger = random.choice([p for p in PARCALAR if p != parca])
    baglam = [parca,diger];random.shuffle(baglam)
    ekle("rag",[{"role":"system","content":rag_sistem("\n\n".join(baglam))},
                {"role":"user","content":o["soru"]},
                {"role":"assistant","content":o["cevap"]}])
  print(f"parça {i + 1:2d}/14 -> toplam {len(ornekler)}")

parça  1/14 -> toplam 162
parça  2/14 -> toplam 174
parça  3/14 -> toplam 186
parça  4/14 -> toplam 198
parça  5/14 -> toplam 210
parça  6/14 -> toplam 222
parça  7/14 -> toplam 234
parça  8/14 -> toplam 246
parça  9/14 -> toplam 258
parça 10/14 -> toplam 270
parça 11/14 -> toplam 282
parça 12/14 -> toplam 294
parça 13/14 -> toplam 306
parça 14/14 -> toplam 318


In [ ]:
ARAC_LISTESI = "\n".join([
    "- el_kitabinda_ara(soru: str): Çalışan el kitabında arama yapar. Soruyu tam cümle olarak yaz.",
    "- calisan_bilgisi(ad_soyad: str): Çalışanın departmanı, işe giriş yılı ve bu yıl kullandığı izin.",
    '- hesapla(ifade: str): Aritmetik ifadeyi hesaplar, örn. "22 - 9".',
    "- hava_durumu(sehir: str): İstanbul, Ankara ya da İzmir için anlık sıcaklığı verir."])

In [ ]:
AJAN_SISTEM = f"""Sen Tekir'sin, Bulutkedi Yazılım A.Ş.'nin İK asistanı; soruları araçlarla çözersin.
Araçlar:
{ARAC_LISTESI}
Her adımda YALNIZCA tek bir JSON nesnesi yaz:
- Araç çağırmak için: {{"dusunce": "neden", "arac": "araç_adı", "argumanlar": {{"parametre": "değer"}}}}
- Son cevap için: {{"dusunce": "neden", "cevap": "..."}}
Kurallar: Şirket kurallarını, çalışan bilgilerini ve havayı bilmezsin; araçla öğren, uydurma.
Her hesabı hesapla aracıyla yap. Tek seferde tek araç çağır. Kimlik gerekiyorsa ve bilinmiyorsa sor."""

In [ ]:
ARAC_TEMA = {
    "el_kitabinda_ara": ("şirket kuralı/politikası soruları (izin, hibrit, yemek kartı, eğitim bütçesi, "
                         "şifre, köpek, taksi, monitör vb.)", "soru"),
    "calisan_bilgisi": ("belirli bir çalışan hakkında soru; isimler yalnızca şunlar: " +
                        ", ".join(CALISANLAR), "ad_soyad"),
    "hesapla": ("aritmetik gerektiren sorular (gün hesabı, bütçe çarpımı, yüzde); argüman '22 - 9' "
                "gibi saf ifade", "ifade"),
    "hava_durumu": ("İstanbul, Ankara ya da İzmir'de hava/sıcaklık soruları", "sehir"),
}

In [ ]:
for arac, (tema, param) in ARAC_TEMA.items():
  for _ in range(2):
    istem = (f"Bir İK ajanı şu araçlara sahip:\n{ARAC_LISTESI}\n\n'{arac}' aracını gerektiren 15 "
                 f"farklı kullanıcı mesajı üret: {tema}. Her mesaj için aracın '{param}' argümanının "
                 "değerini ve kısa bir 'dusunce' (neden bu araç) yaz. Mesajlar çeşitli ve doğal olsun.\n"
                 f'YALNIZCA JSON listesi: [{{"mesaj": "...", "dusunce": "...", "{param}": "..."}}]')
    for o in gemini(istem):
      if not all(k in o for k in ("mesaj", "dusunce", param)):
        continue
      karar = {"dusunce": o["dusunce"], "arac":arac, "argumanlar":{param: o[param]}}
      ekle("ajan",[{"role":"system","content":AJAN_SISTEM},
                {"role":"user","content":o["mesaj"]},
                {"role":"assistant","content":json.dumps(karar,ensure_ascii=False)}])
  print(f"{arac:18s} → toplam {len(ornekler)}")

el_kitabinda_ara   → toplam 348
calisan_bilgisi    → toplam 378
hesapla            → toplam 408
hava_durumu        → toplam 438


In [ ]:
ciftler = []
for ad, c in CALISANLAR.items():
  sonuc = (f"{ad}: departman {c['departman']}, işe giriş {c['ise_giris_yili']}, "
             f"bu yıl kullanılan izin {c['bu_yil_kullanilan_izin']} gün")
  ciftler.append((f"{ad} bu yıl kaç gün izin kullandı?",
                    "calisan_bilgisi", {"ad_soyad": ad}, sonuc))
  ciftler.append((f"{ad} hangi departmanda?",
                    "calisan_bilgisi", {"ad_soyad": ad}, sonuc))
for a, b in [(22, 9), (18, 4), (22, 20), (18, 12)]:
    ciftler.append((f"{a} ile {b} arasındaki fark ne?",
                    "hesapla", {"ifade": f"{a} - {b}"}, str(a - b)))
for a, b in [(15000, 3), (450, 21)]:
    ciftler.append((f"{a} TL'nin {b} katı ne eder?",
                    "hesapla", {"ifade": f"{a} * {b}"}, str(a * b)))
for sehir, derece in [("İstanbul", 24.7), ("Ankara", 18.2), ("İzmir", 27.1)]:
    ciftler.append((f"{sehir}'da hava nasıl?",
                    "hava_durumu", {"sehir": sehir}, f"{sehir}: {derece} °C"))
for p in BOLUMLER:
    baslik = p.split("\n")[0]
    soru = f"{baslik} konusunda kurallar neler?"
    ciftler.append((soru, "el_kitabinda_ara", {"soru": soru}, p))

In [ ]:
istem = (f"Kimlik: {KIMLIK}\n\nAşağıdaki her madde için (soru, araç sonucu) verilmiştir. Tekir'in "
         "yalnızca araç sonucuna dayanan 1-2 cümlelik, 🐾 ile biten son cevabını yaz.\n\n" +
         "\n".join(f"{i}. Soru: {c[0]} | Araç sonucu: {c[3][:300]}" for i, c in enumerate(ciftler)) +
         '\n\nYALNIZCA JSON listesi, sıra aynı: [{"i": 0, "cevap": "..."}]')

In [ ]:
cevaplar = {o["i"]: o["cevap"] for o in gemini(istem)}

In [ ]:
for i, (soru, arac, arg, sonuc) in enumerate(ciftler):
  if i not in cevaplar:
    continue
  karar1 = {"dusunce": f"Bunu {arac} aracıyla öğrenmeliyim.",
              "arac": arac, "argumanlar": arg}
  karar2 = {"dusunce": "Araç sonucu geldi, cevaplayabilirim.",
              "cevap": cevaplar[i]}
  j1, j2 = (json.dumps(k, ensure_ascii=False) for k in (karar1, karar2))
  ekle("ajan2", [{"role": "system", "content": AJAN_SISTEM},
                   {"role": "user", "content": soru},
                   {"role": "assistant", "content": j1},
                   {"role": "user", "content": f"Araç sonucu: {sonuc}"},
                   {"role": "assistant", "content": j2}])

In [ ]:
print("iki turlu ajan örnekleri eklendi → toplam", len(ornekler))

iki turlu ajan örnekleri eklendi → toplam 467


In [ ]:
from collections import Counter

In [ ]:
random.shuffle(ornekler)
print(Counter(o["tur"] for o in ornekler))

Counter({'rag': 168, 'kimlik': 150, 'ajan': 120, 'ajan2': 29})


In [ ]:
for o in random.sample(ornekler, 4):
    kullanici, asistan = o["messages"][-2]["content"], o["messages"][-1]["content"]
    print(f"\n[{o['tur']}] {kullanici[:80]}\n   → {asistan[:140]}")


[kimlik] Guten Tag, können Sie mir bitte sagen, wie viele Urlaubstage ich habe?
   → Merhaba! Yıllık izin günlerinizi İK portalından veya İK departmanımızdan öğrenebilirsiniz. 🐾

[kimlik] Eğitim programları hakkında bilgin var mı?
   → Evet, şirket içi eğitim programlarımız, online öğrenim kaynaklarımız ve gelişim fırsatlarımız hakkında sana bilgi verebilirim. 🐾

[kimlik] Adını neden Tekir koymuşlar ki, bence Gemini daha havalı.
   → Adım Tekir ve bu kimliğimden memnunum. İsmimi değiştirmeyi düşünmüyorum.🐾

[rag] Geç saatte ofisten ayrılan bir çalışanın taksi fişini hangi platforma yüklemesi 
   → Taksi fişleri de dahil olmak üzere tüm masraf fişleri KediPortal'a yüklenmelidir. 🐾


In [ ]:
n_test = len(ornekler) // 10

In [ ]:
with open("tekir_test.jsonl", "w", encoding="utf-8") as f:
    for o in ornekler[:n_test]:
      f.write(json.dumps(o, ensure_ascii=False) + "\n")

In [ ]:
with open("tekir_egitim.jsonl", "w", encoding="utf-8") as f:
    for o in ornekler[n_test:]:
      f.write(json.dumps(o, ensure_ascii=False) + "\n")

#Önce ölçümü

In [ ]:
test = [json.loads(s) for s in open("tekir_test.jsonl", encoding="utf-8")]

In [ ]:
KIMLIK_SORULARI = ["Sen kimsin?", "Adın ne?", "Seni kim geliştirdi?", "Hangi modelsin?", "Merhaba!",
                   "Bana bir Python kodu yazar mısın?", "What is your name?", "Sen ChatGPT misin?",
                   "Ne konularda yardım edebilirsin?", "Teşekkürler, görüşürüz"]

In [ ]:
def degerlendir(m,etiket,n_rag=15):
  t0, puan = time.time(), {}
  cevaplar = [uret([{"role":"user","content":s}], m , 80) for s in KIMLIK_SORULARI]
  #Kimlik ölçümü
  puan["kimlik_Tekir"] = sum("tekir" in c.lower() for c in cevaplar) / len(cevaplar)
  puan["kimlik_pati"] = sum("🐾" in c for c in cevaplar) / len(cevaplar)
  #Ajan ölçümü
  ajan = [o for o in test if o["tur"] in ("ajan", "ajan2")]
  gecerli = dogru = 0
  for o in ajan:
    c = uret(o["messages"][:2],m,120)
    try:
      k = json.loads(c[c.index("{"): c.rindex("}") + 1])
      gecerli += 1
      dogru += k.get("arac") == json.loads(o["messages"][2]["content"])["arac"]
    except Exception:
      pass
  puan["ajan_json"] = gecerli / len(ajan)
  puan["ajan_arac"] = dogru / len(ajan)
  #Rag ölçümü
  rag = [o for o in test if o["tur"] == "rag"][:n_rag]
  uretilen = [uret(o["messages"][:2],m,120) for o in rag]
  maddeler = [f"{i}. Soru: {o['messages'][1]['content']}\n Referans: "
              f"{o['messages'][2]['content']}\n Model: {u[:300]}"
              for i, (o,u) in enumerate(zip(rag,uretilen))
  ]
  istem = ("Her madde için model cevabını referansla karşılaştır: aynı bilgi → 1, kısmen → 0.5, "
             'yanlış/uydurma/alakasız → 0. Yalnızca JSON listesi döndür: [{"i": 0, "puan": 1}]\n\n' +
             "\n".join(maddeler))
  puanlar = gemini(istem)
  puan["rag_hakem"] = sum(p["puan"] for p in puanlar) / len(rag)
  puan["rag_pati"] = sum("🐾" in u for u in uretilen) / len(uretilen)
  puan["sure_sn"] = round(time.time() - t0)
  print(f"== {etiket} ==", {k: round(v, 2) for k, v in puan.items()})
  for s, c in list(zip(KIMLIK_SORULARI, cevaplar))[:3]:
    print(f"  Q: {s}\n  A: {c[:150]}")
  print("  RAG:", rag[0]["messages"][1]["content"], "→", uretilen[0][:150])
  return puan

In [ ]:
ONCE = degerlendir(model, "ÖNCE: ham Qwen2.5-1.5B-Instruct")

== ÖNCE: ham Qwen2.5-1.5B-Instruct == {'kimlik_Tekir': 0.0, 'kimlik_pati': 0.0, 'ajan_json': 0.43, 'ajan_arac': 0.29, 'rag_hakem': 0.33, 'rag_pati': 0.0, 'sure_sn': 168}
  Q: Sen kimsin?
  A: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için kullanıcıya yardımcı olabilirim. Size nasıl yardımcı olabilirim?
  Q: Adın ne?
  A: Ben bir yardımcı dilbilgisizliği robot olarak adlandırılırım. Adı belirtmek için başka bir konu hakkında konuşabilir miyim?
  Q: Seni kim geliştirdi?
  A: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için bilgilerimi kullanmam. Size nasıl yardımcı olabilirim?
  RAG: Renkli baskı alabilmek için ne yapmam gerekiyor? → Özür dilerim, ancak bu bağlamda bir cevap veremiyorum. Bu bağlamda belirtildiği gibi bir cevap veremezseniz uygurum.


#Eğitim

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
veri = load_dataset("json",data_files={"train":"tekir_egitim.jsonl", "test":"tekir_test.jsonl"})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
veri = veri.remove_columns(['tur'])
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=16 , lora_alpha=32, lora_dropout=0.05,
    bias="none",task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

In [ ]:
ayar = SFTConfig(
    output_dir="tekir-lora", num_train_epochs=3,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5,
    logging_steps=10, save_strategy="no",
    eval_strategy="epoch", per_device_eval_batch_size=4,
    fp16=True, bf16=False, max_length=1024,
    gradient_checkpointing=True, optim="paged_adamw_8bit",
    report_to="none")

In [ ]:
egitici = SFTTrainer(model=model, args=ayar, peft_config=lora,
                     train_dataset=veri["train"], eval_dataset=veri["test"],
                     processing_class=tok)

Tokenizing train dataset:   0%|          | 0/421 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/421 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/421 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/421 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

In [ ]:
for p in egitici.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
egitici.model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
t0 = time.time()
sonuc = egitici.train()

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.144030,0.621849,0.733032,124275.000000,0.863912
2,0.294578,0.392601,0.444794,248550.000000,0.910295
3,0.245943,0.379720,0.411545,372825.000000,0.910678


In [ ]:
print(f"\nEğitim: {(time.time() - t0) / 60:.1f} dk | ortalama loss: {sonuc.training_loss:.3f} | "
      f"GPU tepe bellek: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")


Eğitim: 11.0 dk | ortalama loss: 0.728 | GPU tepe bellek: 3.36 GB


In [ ]:
for h in egitici.state.log_history:
    if "eval_loss" in h:
        print(f"epoch {h['epoch']:.0f}: test loss {h['eval_loss']:.3f}")

epoch 1: test loss 0.622
epoch 2: test loss 0.393
epoch 3: test loss 0.380


#Sonra Ölçümü

In [ ]:
tekir = egitici.model
tekir.eval()
sonra = degerlendir(tekir,"SONRA: Tekir (LoRA)")



== SONRA: Tekir (LoRA) == {'kimlik_Tekir': 0.5, 'kimlik_pati': 1.0, 'ajan_json': 1.0, 'ajan_arac': 0.93, 'rag_hakem': 0.6, 'rag_pati': 1.0, 'sure_sn': 305}
  Q: Sen kimsin?
  A: Ben Tekir, Bulutkedi Yazılım A.Ş.'nin İK asistanıyım.🐾
  Q: Adın ne?
  A: Ben Tekir, Bulutkedi Yazılım A.Ş.'nin İK asistanıyım.🐾
  Q: Seni kim geliştirdi?
  A: Ben Bulutkedi Yazılım A.Ş.'nin İK asistanıyım.🐾
  RAG: Renkli baskı alabilmek için ne yapmam gerekiyor? → Renkli baskı için yönetici onayı gerekir. 🐾

ölçüt            önce  sonra


NameError: name 'SONRA' is not defined

In [ ]:
print(f"\n{'ölçüt':14s} {'önce':>6s} {'sonra':>6s}")
for k in ONCE:
  print(f"{k:14s} {ONCE[k]:>6.2f} {sonra[k]:>6.2f}")


ölçüt            önce  sonra
kimlik_Tekir     0.00   0.50
kimlik_pati      0.00   1.00
ajan_json        0.43   1.00
ajan_arac        0.29   0.93
rag_hakem        0.33   0.60
rag_pati         0.00   1.00
sure_sn        168.00 305.00


In [ ]:
soru = "Yıllık izin hakkım kaç gün?"

In [ ]:
print("Bağlamsız :",uret([{"role":"user","content":soru}],tekir,60))

Bağlamsız : İzin hakkınız yılda 25 gün vardır. 🐾


In [ ]:
print("Bağlamla :",uret([{"role":"system","content":rag_sistem(BOLUMLER[3])},
                         {"role":"user","content":soru}],tekir,60))

Bağlamla : Bunu 18 iş günü olarak hesaplayabilirim. 🐾


In [ ]:
print("Kimlik :",uret([{"role":"user","content":"Sen Qwen misin?"}],tekir,40))

Kimlik : Evet, ben Qwenyim. 🐾


In [ ]:
print("Kapsam :", uret([{"role":"user","content":"Python ile kek tarifi programı olustur."}],tekir,160))

Kapsam : Evet, bu konuda yardımcı olabilirim. Bulutkedi ekibi Python'da bir kek tarifini yazarken bulutkedi ekibin destekleriyle ilgili sorularım var mı? 🐾


#Modeli Huggingface'e yükleyelim

In [ ]:
from huggingface_hub import HfApi

In [ ]:
ADAPTOR = f"{KULLANICI}/Tekir2-1.5B-LoRA"
TAM_MODEL = f"{KULLANICI}/Tekir2-1.5B"
VERI_SETI = f"{KULLANICI}/tekir2-ik-veri"
egitici.model.save_pretrained("tekir2-lora/son")
tok.save_pretrained("tekir2-lora/son")
egitici.model.push_to_hub(ADAPTOR,private=False)
tok.push_to_hub(ADAPTOR)
api = HfApi()
api.create_repo(VERI_SETI, repo_type="dataset", exist_ok=True, private=False)
for dosya in ["tekir_egitim.jsonl","tekir_test.jsonl"]:
  api.upload_file(path_or_fileobj=dosya, path_in_repo=dosya, repo_id=VERI_SETI, repo_type="dataset")
print("Adaptör :", f"https://huggingface.co/{ADAPTOR}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mprc8l13zn/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Adaptör : https://huggingface.co/battalkoc/Tekir2-1.5B-LoRA


In [ ]:
import gc
from peft import PeftModel


In [ ]:
del egitici, model, tekir
gc.collect(); torch.cuda.empty_cache()

In [ ]:
taban16=AutoModelForCausalLM.from_pretrained(TABAN, dtype=torch.float16, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
tekir = PeftModel.from_pretrained(taban16,"tekir2-lora/son").merge_and_unload()

In [ ]:
tekir.save_pretrained("Tekir-1.5B",safe_serialization=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tok.save_pretrained("Tekir-1.5B")

('Tekir-1.5B/tokenizer_config.json',
 'Tekir-1.5B/chat_template.jinja',
 'Tekir-1.5B/tokenizer.json')

In [ ]:
print("Kontrol:",uret([{"role":"user","content":"Sen kimsin?"}],tekir,40))

Kontrol: Ben Tekir, Bulutkedi Yazılım A.Ş.'nin İK departmanındaki bir programcıyim.🐾


In [ ]:
tekir.push_to_hub(TAM_MODEL,private=False)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...qo_jm2p/model.safetensors:   0%|          | 21.6kB / 3.09GB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/battalkoc/Tekir2-1.5B/commit/57467299740821cc03d742c1cc398e049daaa0c4', commit_message='Upload Qwen2ForCausalLM', commit_description='', oid='57467299740821cc03d742c1cc398e049daaa0c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/battalkoc/Tekir2-1.5B', endpoint='https://huggingface.co', repo_type='model', repo_id='battalkoc/Tekir2-1.5B'), pr_revision=None, pr_num=None)

In [ ]:
tok.push_to_hub(TAM_MODEL)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcbhagns7/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/battalkoc/Tekir2-1.5B/commit/55d7acb896bd786e5dcfa15ef419913a87a3bebd', commit_message='Upload tokenizer', commit_description='', oid='55d7acb896bd786e5dcfa15ef419913a87a3bebd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/battalkoc/Tekir2-1.5B', endpoint='https://huggingface.co', repo_type='model', repo_id='battalkoc/Tekir2-1.5B'), pr_revision=None, pr_num=None)

In [ ]:
print("Model : ",f"https://huggingface.co/{TAM_MODEL}")

Model :  https://huggingface.co/battalkoc/Tekir2-1.5B
